# Ep. 02 — Tokens, embeddings y un pipeline determinista

Los LLM no “ven” el texto como caracteres o palabras. Ven **tokens**. Los embeddings proyectan texto en vectores numéricos para comparar significado. Al final armamos un pipeline en clases que interpreta un **ACR** (relato/caso clínico) y devuelve JSON estructurado.


## 1. Tokenización

Limitación: contar **palabras** ≠ contar **tokens**. Espacios y puntuación suelen pegarse al trozo anterior. Abajo, un texto de ejemplo pasa por el tokenizer BPE `cl100k_base` (familia GPT-4).


In [ ]:
import tiktoken

text = "La inteligencia artificial procesa texto en tokens, no en palabras."
enc = tiktoken.get_encoding("cl100k_base")
token_ids = enc.encode(text)
tokens = [enc.decode([tid]) for tid in token_ids]

print("Texto:", text)
print("Tokens:", tokens)
print("Cantidad de tokens:", len(tokens))
print("Cantidad de palabras (split simple):", len(text.split()))


## 2. Embeddings (Hugging Face / sentence-transformers)

Un embedding es un vector denso que representa el significado de un texto. Usamos el modelo ligero `sentence-transformers/all-MiniLM-L6-v2` (la primera ejecución descarga el modelo).


In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

word = "alegría"
embedding = model.encode(word)

print("Palabra:", word)
print("Dimensión del embedding:", len(embedding))
print("Primeros 8 valores:", embedding[:8])


## 3. Comparación: significados opuestos

Dos textos con sentidos opuestos quedan en regiones distintas del espacio vectorial. Medimos **similitud de coseno** (1 ≈ mismo sentido; valores bajos ≈ lejos).


In [ ]:
import numpy as np

w1, w2 = "alegría", "tristeza"
e1, e2 = model.encode([w1, w2])

def cosine_similarity(a, b):
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

def euclidean_distance(a, b):
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    return float(np.linalg.norm(a - b))

sim = cosine_similarity(e1, e2)
dist = euclidean_distance(e1, e2)

print(f"{w1!r} vs {w2!r}")
print("Similitud de coseno:", round(sim, 4))
print("Distancia euclidiana:", round(dist, 4))


## 4. Visualización 3D (PCA)

Los embeddings tienen cientos de dimensiones. Para **visualizar**, reducimos a 3D con PCA (solo para el gráfico — la similitud real usa el vector completo).


In [ ]:
from sklearn.decomposition import PCA
import plotly.graph_objects as go

labels = ["alegría", "tristeza", "felicidad", "ira", "calma"]
vectors = model.encode(labels)
coords = PCA(n_components=3).fit_transform(vectors)

fig = go.Figure(
    data=[
        go.Scatter3d(
            x=coords[:, 0],
            y=coords[:, 1],
            z=coords[:, 2],
            mode="markers+text",
            text=labels,
            textposition="top center",
            marker=dict(size=6),
        )
    ]
)
fig.update_layout(
    title="Embeddings en 3D (PCA)",
    scene=dict(xaxis_title="PC1", yaxis_title="PC2", zaxis_title="PC3"),
    width=700,
    height=500,
)
fig.show()


## 5. Pipeline determinista: interpretar un ACR

Aquí un **ACR** es un relato/caso clínico corto que el estudiante puede editar. Dos clases en secuencia: `LLMClient` hace la llamada HTTP a la API NVIDIA NIM (formato OpenAI-compatible) y `ACRInterpreter` pide JSON estructurado (`category`, `summary`, `urgency`). Sin LangChain — solo `urllib` + `NVIDIA_API_KEY`.


In [ ]:
import os
import json
import urllib.request


class LLMClient:
    """Llamada vanilla a la API NVIDIA NIM (formato OpenAI-compatible)."""

    def __init__(self):
        self.api_key = os.environ.get("NVIDIA_API_KEY", "").strip()
        self.model = os.environ.get("NVIDIA_MODEL", "meta/llama-3.1-8b-instruct")

    def ask(self, messages):
        if isinstance(messages, str):
            messages = [{"role": "user", "content": messages}]
        if not self.api_key:
            raise SystemExit("Define NVIDIA_API_KEY y vuelve a ejecutar.")
        body = json.dumps(
            {
                "model": self.model,
                "messages": messages,
                "temperature": 0,
                "response_format": {"type": "json_object"},
            }
        ).encode()
        req = urllib.request.Request(
            "https://integrate.api.nvidia.com/v1/chat/completions",
            data=body,
            headers={
                "Authorization": f"Bearer {self.api_key}",
                "Content-Type": "application/json",
            },
            method="POST",
        )
        with urllib.request.urlopen(req) as resp:
            data = json.loads(resp.read().decode())
        return data["choices"][0]["message"]["content"]


class ACRInterpreter:
    """Una clase llama a la otra: interpreta el ACR vía LLMClient."""

    SYSTEM = (
        "Eres un asistente clínico. Dado un ACR (relato de caso), "
        "responde SOLO con JSON que contenga las claves: "
        "category (string), summary (string corta), urgency (baja|media|alta)."
    )

    def __init__(self, llm: LLMClient):
        self.llm = llm

    def interpret(self, acr: str) -> dict:
        raw = self.llm.ask(
            [
                {"role": "system", "content": self.SYSTEM},
                {"role": "user", "content": acr},
            ]
        )
        return json.loads(raw)


# Edita el ACR abajo y ejecuta la celda
acr = (
    "Paciente de 67 años, hipertenso, llega con dolor torácico opresivo "
    "desde hace 40 minutos, sudoración y disnea. PA 90/60 mmHg."
)

result = ACRInterpreter(LLMClient()).interpret(acr)
print(json.dumps(result, ensure_ascii=False, indent=2))


## 6. Docling + OCR (demostración)

Los PDF escaneados a menudo no tienen capa de texto fiable — ahí entra el **OCR**. Docling encapsula el pipeline: `PdfPipelineOptions(do_ocr=True)` + `DocumentConverter`. La celda de abajo es **solo demostración**: deja `RUN_DOCLING = False` salvo que tengas un PDF/imagen local y las dependencias de OCR instaladas.


In [ ]:
# Demostración — NO ejecutes a escala sin un archivo local.
# Define RUN_DOCLING = True y una ruta válida para probar.

RUN_DOCLING = False
PDF_PATH = "tu-archivo-escaneado.pdf"  # PDF o imagen

if not RUN_DOCLING:
    print("Demo Docling apagada. Define RUN_DOCLING=True y PDF_PATH para extraer texto.")
else:
    from docling.datamodel.base_models import InputFormat
    from docling.datamodel.pipeline_options import PdfPipelineOptions
    from docling.document_converter import DocumentConverter, PdfFormatOption

    pipeline_options = PdfPipelineOptions(do_ocr=True)
    converter = DocumentConverter(
        format_options={
            InputFormat.PDF: PdfFormatOption(pipeline_options=pipeline_options),
        }
    )
    result = converter.convert(PDF_PATH)
    text = result.document.export_to_markdown()
    print(text[:2000])


## 7. LM Studio local (Nemotron 3 Nano 4B) — solo demostración

Para esta demo necesitas tener **LM Studio** en ejecución localmente con el modelo **Nemotron 3 Nano 4B** cargado.
La base local por defecto (API compatible con el formato OpenAI — LM Studio) es `http://localhost:1234/v1`.

Esta celda **no forma parte de la lección estándar** — solo demostración. Mantén `RUN_LM_STUDIO = False` salvo que LM Studio esté activo con el modelo cargado.
Comprueba el **model id** exacto en LM Studio (placeholder abajo: `nvidia/Nemotron-Nano-4B`) y ajústalo si hace falta.


In [ ]:
# Demostración — NO forma parte de la lección estándar.
# Define RUN_LM_STUDIO = True con LM Studio en marcha y Nemotron 3 Nano 4B cargado.

RUN_LM_STUDIO = False
LM_STUDIO_BASE = "http://localhost:1234/v1"
MODEL_ID = "nvidia/Nemotron-Nano-4B"  # ajusta al id exacto en LM Studio

if not RUN_LM_STUDIO:
    print("Demo LM Studio apagada. Define RUN_LM_STUDIO=True con el servidor local activo.")
else:
    import json
    import urllib.request

    payload = {
        "model": MODEL_ID,
        "messages": [
            {"role": "user", "content": "Di hola en una frase corta."},
        ],
        "temperature": 0.2,
        "max_tokens": 64,
    }
    req = urllib.request.Request(
        f"{LM_STUDIO_BASE}/chat/completions",
        data=json.dumps(payload).encode("utf-8"),
        headers={"Content-Type": "application/json"},
        method="POST",
    )
    with urllib.request.urlopen(req, timeout=60) as resp:
        data = json.loads(resp.read().decode("utf-8"))
    print(data["choices"][0]["message"]["content"])
